In [ ]:
def evaluate_model(model, generator, tag):
    generator.reset()
    probs = model.predict(generator, verbose=1)
    y_pred = np.argmax(probs, axis=1)
    y_true = generator.classes
    classes = list(generator.class_indices.keys())
    y_conf = probs.max(axis=1)

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    print(f"\n=== {tag} — Test Set Metrics ===")
    print(f"Accuracy       : {acc:.4f}")
    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall   : {recall:.4f}")
    print(f"Macro F1       : {f1:.4f}\n")
    report_str = classification_report(y_true, y_pred, target_names=classes, zero_division=0)
    print(report_str)

    out_dir = RESULTS_DIR / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump({"model": tag, "accuracy": acc, "macro_precision": precision,
                    "macro_recall": recall, "macro_f1": f1}, f, indent=2)
    with open(out_dir / "classification_report.txt", "w") as f:
        f.write(report_str)

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    n_classes = len(classes)
    fig_size = max(11, n_classes * 0.75)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))

    # Annotate each cell with "row-normalised %\n(raw count)" so the figure is
    # readable on its own, without needing the raw cm printed separately.
    annot = np.empty_like(cm).astype(object)
    for i in range(n_classes):
        for j in range(n_classes):
            pct = cm_norm[i, j] * 100
            annot[i, j] = f"{pct:.0f}%\n({cm[i, j]})" if cm[i, j] > 0 else ""

    sns.heatmap(
        cm_norm, ax=ax, cmap="Blues", vmin=0, vmax=1,
        xticklabels=classes, yticklabels=classes,
        annot=annot, fmt="", annot_kws={"size": 9},
        linewidths=0.5, linecolor="white",
        cbar_kws={"label": "Fraction of true class (row-normalised)"},
        square=True,
    )

    ax.set_xlabel("Predicted label", fontsize=13, labelpad=10)
    ax.set_ylabel("True label", fontsize=13, labelpad=10)
    ax.set_title(
        f"Confusion Matrix (row-normalised) — {tag}\n"
        f"Accuracy: {acc:.1%}  |  Macro-F1: {f1:.3f}",
        fontsize=14, pad=14,
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
    ax.tick_params(left=False, bottom=False)

    plt.tight_layout()
    plt.savefig(out_dir / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.show()

    return dict(tag=tag, acc=acc, precision=precision, recall=recall, f1=f1,
                cm=cm, classes=classes, y_true=y_true, y_pred=y_pred, y_conf=y_conf,
                filenames=generator.filenames)

baseline_eval = evaluate_model(baseline_cnn, test_generator, tag="baseline")
transfer_eval = evaluate_model(transfer_model, transfer_test_generator, tag="transfer")


Error analysis


In [ ]:
best_eval = transfer_eval if transfer_eval["f1"] >= baseline_eval["f1"] else baseline_eval
print(f"Running detailed error analysis on: {best_eval['tag']} (macro-F1={best_eval['f1']:.3f})")

classes = best_eval["classes"]
cm = best_eval["cm"].copy()
np.fill_diagonal(cm, 0)
confused_pairs = []
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i, j] > 0:
            confused_pairs.append((classes[i], classes[j], int(cm[i, j])))
confused_pairs.sort(key=lambda x: -x[2])

print("\nTop 10 most-confused class pairs (true -> predicted, count):")
for true_c, pred_c, n in confused_pairs[:10]:
    print(f"  {true_c:20s} -> {pred_c:20s} : {n}")

out_dir = RESULTS_DIR / best_eval["tag"]
with open(out_dir / "confused_pairs.json", "w") as f:
    json.dump(confused_pairs[:20], f, indent=2)


In [ ]:
# Grid of the most confidently WRONG predictions
y_true, y_pred, y_conf = best_eval["y_true"], best_eval["y_pred"], best_eval["y_conf"]
filenames = best_eval["filenames"]

wrong_idx = [i for i in range(len(y_true)) if y_pred[i] != y_true[i]]
wrong_idx.sort(key=lambda i: -y_conf[i])
top_wrong = wrong_idx[:24]

if top_wrong:
    cols = 6
    rows = (len(top_wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.5, rows*2.8))
    axes = axes.flatten()
    for ax, i in zip(axes, top_wrong):
        img_path = TEST_DIR / filenames[i]
        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            pass
        ax.set_title(f"T:{classes[y_true[i]]}\nP:{classes[y_pred[i]]} ({y_conf[i]:.2f})", fontsize=7)
        ax.axis("off")
    for ax in axes[len(top_wrong):]:
        ax.axis("off")
    plt.suptitle(f"Most confidently WRONG predictions — {best_eval['tag']} (T=true, P=predicted)")
    plt.tight_layout()
    plt.savefig(out_dir / "error_analysis_grid.png", dpi=150)
    plt.show()
else:
    print("No misclassifications on the test set (unlikely, but check evaluation code if this happens).")
